# ERICA end-to-end walkthrough

A complete tour of the `erica` library on the VDX 3-gene breast-cancer expression dataset (344 samples × 3 genes). Mirrors `end_to_end_demo.py` cell-for-cell but with narrative.

**What you'll see:**
1. Load data and run ERICA with K-Means + Agglomerative Ward across K = 2..6.
2. Replicability metrics (CRI / WCRI / TWCRI) per method, vs K.
3. K* selected per (metric, method) as a grouped bar chart.
4. CLAM matrix at K* — heatmaps (sorted and unsorted), per-cluster scatter (PCSP), inter-cluster assignment heatmap (ICAH), per-sample stability strips, and cluster sizes.

**Prerequisite:** `pip install erica` (installs all plotting deps).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from erica import (
    ERICA,
    set_publication_style,
    plot_replicability_metrics,
    plot_metrics_vs_k,
    plot_clam_heatmap_mpl,
    plot_cluster_sizes_mpl,
    plot_icah,
    plot_pcsp,
    plot_k_star_bars,
    plot_stability_strips,
    extract_metric_curves,
)

set_publication_style()
%matplotlib inline

## 1. Load the VDX 3-gene dataset

344 samples (rows), 3 genes (columns). ERICA expects `(n_samples, n_features)`.

In [ ]:
CSV_PATH = os.path.join('..', 'examples', 'data', 'VDX_3_SV.csv')
X = pd.read_csv(CSV_PATH, header=None).to_numpy()
print(f'shape: {X.shape}  (n_samples, n_features)')
print(f'first 3 rows:\n{X[:3]}')

## 2. Run ERICA

Monte Carlo subsampling — 50 train/test iterations per K, two clustering methods. The CLAM matrix is built per `(K, method)` and CRI/WCRI/TWCRI derived from it.

In [ ]:
er = ERICA(
    data=X,
    k_range=[2, 3, 4, 5, 6],
    n_iterations=50,
    method=['kmeans', 'agglomerative_ward'],
    transpose=False,
    random_state=42,
    verbose=False,
)
results = er.run()

k_star_twcri = er.get_k_star('TWCRI')
print(f'K* (TWCRI) per method: {k_star_twcri}')

## 3. Replicability metrics (single method)

CRI / WCRI / TWCRI vs K for K-Means. The vertical dashed line marks K* selected by Algorithm 2 on TWCRI.

In [ ]:
method = 'kmeans'
k = k_star_twcri.get(method) or 3

k_values, curves, _ = extract_metric_curves(results, method, ['CRI', 'WCRI', 'TWCRI'])
fig, ax = plot_replicability_metrics(k_values, curves['CRI'], curves['WCRI'], curves['TWCRI'])
ax.axvline(k, color='gray', linestyle='--', linewidth=0.8, label=f'K*={k}')
ax.legend()
ax.set_title(f'VDX 3-gene — {method} replicability metrics')
plt.show()

## 4. Method comparison (side by side)

Same metrics for each method on its own axis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, m in zip(axes, ['kmeans', 'agglomerative_ward']):
    kvals, c, _ = extract_metric_curves(results, m, ['CRI', 'WCRI', 'TWCRI'])
    plot_metrics_vs_k(kvals, c, ax=ax)
    ax.set_title(m)
fig.suptitle('Method comparison — CRI / WCRI / TWCRI vs K', fontsize=12)
fig.tight_layout()
plt.show()

## 5. K* by method and metric

Grouped bar chart — the 'compare methods at a glance' figure. Bar tops annotated with K*.

In [ ]:
k_star_all = {m: er.get_k_star(m) for m in ['CRI', 'WCRI', 'TWCRI']}
fig, _ = plot_k_star_bars(k_star_all)
plt.show()

## 6. CLAM heatmap at K*

Each row is a sample, each column a cluster. Cell value = number of iterations that sample was assigned to that cluster (out of `n_iterations` it appeared in the test split). Sorted version groups samples by primary cluster.

In [ ]:
clam = results['clam_matrices'][(k, method)]

fig, _ = plot_clam_heatmap_mpl(clam, sort=True, title=f'CLAM heatmap (sorted) — K={k}')
plt.show()
fig, _ = plot_clam_heatmap_mpl(clam, sort=False, title=f'CLAM heatmap (unsorted) — K={k}')
plt.show()

## 7. Per-sample stability strips

Each row = one sample. Bar segments = proportion of iterations the sample went to each cluster. Sorted by Shannon entropy, most stable on top. Bottom rows are samples that drift between clusters.

In [ ]:
fig, _ = plot_stability_strips(clam, max_samples=80, title=f'Stability strips — K={k}')
plt.show()

## 8. Inter-Cluster Assignment Heatmap (ICAH)

K×K matrix. Cell `[i, j]` = mean normalized assignment of cluster-i samples to cluster j. Diagonal is self-assignment (high = good); off-diagonal is leakage between clusters.

In [ ]:
fig, _ = plot_icah(clam, k=k, title=f'Inter-Cluster Assignment Heatmap — K={k}')
plt.show()

## 9. Per-Cluster Scatter Plots (PCSP)

For each cluster, a scatter of normalized assignment values across the *other* clusters. Tight low-value clouds = clean separation; spread-out values = ambiguous samples.

In [ ]:
fig = plot_pcsp(clam, k=k, title=f'Per-Cluster Scatter Plots — K={k}')
plt.show()

## 10. Cluster sizes at K*

Bar chart of how many samples are primarily assigned to each cluster (argmax over CLAM rows).

In [ ]:
fig, _ = plot_cluster_sizes_mpl(clam)
plt.show()

## Recap

- ERICA fit + analysis: ~10 lines.
- Every plot above used a single library function on either `clam` (CLAM matrix) or `results` (the ERICA result dict).
- All primitives accept `ax=None` so you can compose them into your own grids.

Next: try your own dataset by replacing `X` in cell 2.